# Pokémon Index

Codetto notebooks can reach out to the internet using Python's `httpx` library, just like a real Python program running on a server. That means students can pull in live, real-world data, e.g., weather, sports scores, trivia, and images instead of only working with data you've typed in by hand.

In this demo we'll use the free [PokéAPI](https://pokeapi.co/), which has structured data (and artwork, and even sound effects) for every Pokémon.

## Our first HTTP GET request

`httpx.get(url)` sends a request and returns a `Response`. `response.json()` turns the reply into a Python dictionary we can pull fields out of.

In [ ]:
import httpx

response = httpx.get("https://pokeapi.co/api/v2/pokemon/ditto")
print(f"Status code: {response.status_code}")

# The full response has dozens of fields - here are a few interesting ones
data = response.json()
print(f"Name: {data['name'].title()}")
print(f"Height: {data['height'] / 10} m")
print(f"Weight: {data['weight'] / 10} kg")
print(f"Abilities: {', '.join(a['ability']['name'] for a in data['abilities'])}")

## Making it interactive

A single cell can combine several API calls into one live tool. Type any Pokémon's name into the field below and click **Run** — Codetto fetches its stats, full-resolution artwork, and even its cry sound, all with `httpx`.

In [ ]:
#@title 🔍 Pokedex Lookup
POKEMON_NAME = "pikachu" #@param

import base64
import httpx
from codetto import audio, graphics

try:
  response = httpx.get(f"https://pokeapi.co/api/v2/pokemon/{POKEMON_NAME.strip().lower()}")
  response.raise_for_status()
except httpx.HTTPStatusError:
  print(f"Couldn't find a Pokemon named '{POKEMON_NAME}'. Check the spelling and try again!")
else:
  data = response.json()
  pokemon_id = data["id"]

  name = data["name"].title()
  height_m = data["height"] / 10
  weight_kg = data["weight"] / 10
  types = ", ".join(t["type"]["name"].title() for t in data["types"])

  print(f"{name}  (#{pokemon_id})")
  print(f"Type: {types}")
  print(f"Height: {height_m} m")
  print(f"Weight: {weight_kg} kg")

  # Full-resolution official artwork
  artwork_url = (
      "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/"
      f"pokemon/other/official-artwork/{pokemon_id}.png"
  )
  artwork = httpx.get(artwork_url)
  artwork_data_url = f"data:image/png;base64,{base64.b64encode(artwork.content).decode()}"
  graphics.display_image(artwork_data_url)

  # Play the Pokemon's cry
  cry_url = (
      "https://raw.githubusercontent.com/PokeAPI/cries/main/cries/"
      f"pokemon/latest/{pokemon_id}.ogg"
  )
  cry = httpx.get(cry_url)
  if cry.status_code == 200:
      with open("/tmp/cry.ogg", "wb") as f:
          f.write(cry.content)
      audio.play("/tmp/cry.ogg")

## Where this could go in your classroom

Once students can call an API, a huge range of projects open up: a weather dashboard, a "random trivia fact" button, a sports score tracker, or combining an API with `scene3d`/`graphics` to visualize live data. The pattern is always the same three steps: **fetch → parse → use the data.**